In [ ]:
%matplotlib widget
# Boilerplate import code for all libraries
# Changes to the precision require re-loading the kernel and need to be done before any op uses them.
import sphWarpCore_config as swc
from typing import Any
swc.configure(precision="float32", dim=Any) # precision: float16|half|float32|single|float64|double

import sphWarpCore as sph
from sphWarpCore.type_config import *
print(get_type_config()) # confirms active settings

# Initialize warp at this point
import warp as wp; wp.init()

import os
import torch
if torch.cuda.is_available(): # set the TORCH_CUDA_ARCH_LIST environment variable to the compute capability of the GPU for faster compiles
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from tqdm.autonotebook import tqdm

# final import blocks that are generic
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
import numpy as np
import math
import shlex    
import subprocess
import shutil

# custom SPH libraries
from integrators.integration import *
from sphWarpCore import *

# This library
from compressibleSPH import *
from compressibleSPH.modules.timestep.compressible import computeTimestep

# The case utilities that contain all the case setup functions for the various test cases
from compressibleSPH.caseUtils import *

{'scalar_t': <class 'warp._src.types.float32'>, 'dim_t': typing.Any}
Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX PRO 500 Blackwell Generation Laptop GPU" (6 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/lu26029/.cache/warp/1.12.0


In [ ]:
nx = 256
dim = 2

L = 1
dx = L/nx
aspect = 2
band = 20

n_h = 4

gamma = 1.4
rho0 = 1.0

rho_I = 1.0
p_I = 1.0
rho_II = 0.125
p_II = 0.1
rho_III = 1.0
p_III = 0.1



extraData = {
    'nx': nx,
    'dim': dim,
    'L': L,
    'n_h': n_h,

    'gamma': gamma,
    'rho0': rho0,
    'rho_I': rho_I,
    'p_I': p_I,
    'rho_II': rho_II,
    'p_II': p_II,
    'rho_III': rho_III,
    'p_III': p_III,

    'dx': dx,
    'aspect': aspect,
    'band': band

}

In [3]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
dtype = get_torch_precision()


domain = buildDomainDescription(l = 1, dim = dim, periodic = True, device = device, dtype = dtype)
domain.min[0] = 0
domain.max[0] = 14
domain.min[1] = -3
domain.max[1] = 3

config, integrator = buildConfig(
    domain = domain,
    dim = dim,
    kernel = KernelFunctions.B7,
    targetNeighbors = n_h_to_nH(4, dim),
    supportMode = SupportScheme.KernelMeanSymmetric,
    gradientMode = GradientScheme.Difference,
    laplacianMode = LaplacianScheme.Brookshaw,
    integrationScheme = IntegrationSchemeType.rungeKutta2,
    samplingScheme = SamplingScheme.regular,
    device = device,
    dtype = dtype,
    dt = None,
    adaptiveDt = True,
    cflFactor=0.3,
)
config.nx = nx

config.minDt = 1e-8
# config.dx = L / (nx * 2)

scheme = CompressibleSPHScheme.CRKSPH
SimulationSystem, SimulationState, SimulationConfig, SimulationUpdate, fn, export_fn, import_fn = buildScheme(scheme)


schemeConfig = SimulationConfig()
schemeConfig.gamma = gamma
schemeConfig.rho0 = rho0


schemeConfig.viscositySwitchParams.scheme = ViscositySwitch.NoneSwitch
schemeConfig.adaptiveSupportScheme = AdaptiveSupportScheme.Owen
schemeConfig.adaptiveSupportCorrections = False

In [4]:
compressibleSystem = sampleTriplePointEqualResolution(
    splitX = 1.0,
    splitY = 1.5,
    rho_I = rho_I,
    p_I = p_I,
    rho_II = rho_II,
    p_II = p_II,
    rho_III = rho_III,
    p_III = p_III,
    nx = nx,
    config = config,
    schemeConfig = schemeConfig,
    extraData = extraData,
    SimulationState = SimulationState,
    SimulationSystem = SimulationSystem
)

Module compressibleSPH.modules.adaptiveSupport.wp_psi 8875fe6 load on device 'cuda:0' took 1.09 ms  (cached)
Module sphWarpCore.radiusSearch.wp_compactHash e67fccd load on device 'cuda:0' took 2.01 ms  (cached)
Module compressibleSPH.modules.adaptiveSupport.wp_psi0 37e5819 load on device 'cuda:0' took 3.22 ms  (cached)
Module sphWarpCore.operations.wp_density f0357bf load on device 'cuda:0' took 2.97 ms  (cached)


In [5]:
runningState = compressibleSystem.initializeNewState()

kineticEnergy = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
thermalEnergy = (runningState.state.internalEnergies * runningState.state.masses).sum()
totalEnergy = kineticEnergy + thermalEnergy

In [6]:
caseName = '14-Triple_point'
exportPath = prepExport(f'{caseName}', config, schemeConfig, scheme, export_fn)
exportSimulationSystem(exportPath, 'initialState', scheme, compressibleSystem, exportAdjacency = False, stages = None, exportStagesAdjacency = False, extraData = dict({
    'kineticEnergy': kineticEnergy,
    'thermalEnergy': thermalEnergy,
    'totalEnergy': totalEnergy,
    'frame_num': 0,
}, **extraData))


In [7]:
from warpPlot import visualize, PlottingOptions, PlotScaling, GridVisualization, UniformColorMap, DivergingColorMap, Mapping, CyclicColorMap
markerSize = 1
plotter = visualize(
    particleState = runningState.state,
    domain = config.domain,
    quantities = {
        "A": runningState.state.densities,
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = DivergingColorMap.RdBu,
            flipColorMap = True,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Logarithmic,
            plotTitle = "density",
            gridVisualization = GridVisualization(
                resolution = 1024,
            ),
            vMin=0.2,
            vMax=7
        ),
    },
    figTitle = "Wave Equation Example",
    mosaic = 'A',
    figsize= (12,6),
    backend='vispy',
)

imagePath = f'{exportPath}/images'
os.makedirs(imagePath, exist_ok = True)
plotter.export(f'{imagePath}/frame_00000.png', dpi = 300)
# if args.exportImages:

RFBOutputContext()

Module sphWarpCore.operations_grid.wp_interpolate_grid bde6e7c load on device 'cuda:0' took 9.63 ms  (cached)


In [12]:
config.cflFactor = 0.2

In [13]:
# config.dt = 2.5e-3
config.dt = computeTimestep(runningState, config, schemeConfig, dt = None) #* 2/3
config.dt = 5.0e-4

t_limit = 10.0
nSteps = int(t_limit / config.dt)

print(f"Running with dt: {config.dt}, which gives nSteps: {nSteps}")
# nSteps = 256

runningState = compressibleSystem.initializeNewState()

trajectory = []

priorStep = None
i = 0
t = 0
tq = tqdm(total = 1000, leave = True)

while t < t_limit:

    begin = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    begin.record()
    result = integrator.function(
        state = runningState,
        f = fn,
        dt = config.dt,  
        config = config,
        compParams = schemeConfig,
        verbose = False,
        # priorStep = priorStep
    )
    end.record()
    torch.cuda.synchronize()
    priorStep = result.stages[-1]
    timing = begin.elapsed_time(end)

    runningState = result.state
    kineticEnergy = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
    thermalEnergy = (runningState.state.internalEnergies * runningState.state.masses).sum()
    totalEnergy = kineticEnergy + thermalEnergy

    trajectory.append(
        (i, runningState.t, config.dt, totalEnergy.item(), kineticEnergy.item(), thermalEnergy.item(), timing)
,     )
    # config.dt = computeTimestep(runningState, config, schemeConfig, dt = config.dt) #* 2/3

    i = i + 1
    t = runningState.t

    if i % 10 == 0 and i > 0:
        plotter.updateQuantities(
            {
                # "A": runningState.state.velocities,
                "A": runningState.state.densities,
            },
            newParticleState = runningState.state,
        )
        plotter.export(f'{imagePath}/frame_{i:05d}.png', dpi = 300)
        
    if i % 500 == 0:
        exportSimulationSystem(exportPath, f'state_{i:04d}', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**extraData, **{
            'kineticEnergy': kineticEnergy,
            'thermalEnergy': thermalEnergy,
            'totalEnergy': totalEnergy,
            'frame_num': i,
        }))

        
    maxVel = torch.linalg.norm(runningState.state.velocities, dim = -1).max()
    tq.set_description(f"Step {i+1}/{nSteps}, time: {(i+1)*config.dt:8.4g}/{t_limit:8.4g}, TE: {totalEnergy:.3g}, KE: {kineticEnergy:.3g}, IE: {thermalEnergy:.3g} | max vel: {maxVel:.3g} | iter time: {timing:.3f} ms")
    tq.n = int(t / t_limit * 1000)
    # t = {runningState.t:2f}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}'
    # break
    torch.cuda.empty_cache()

Running with dt: 0.0005, which gives nSteps: 20000


  0%|          | 0/1000 [00:00<?, ?it/s]

Module sphWarpCore.crk.crk_volume 793e787 load on device 'cuda:0' took 2.56 ms  (cached)
Module sphWarpCore.crk.crk_moments 89bc989 load on device 'cuda:0' took 4.32 ms  (cached)
Module sphWarpCore.crk.crk_density 06a7669 load on device 'cuda:0' took 2.13 ms  (cached)
Module sphWarpCore.operations.wp_gradient a50c471 load on device 'cuda:0' took 4.87 ms  (cached)
Module compressibleSPH.modules.crk.accel 9056830 load on device 'cuda:0' took 4.09 ms  (cached)
Module compressibleSPH.modules.crk.dudt 7e9ded3 load on device 'cuda:0' took 4.01 ms  (cached)
Module compressibleSPH.modules.compSPH.balance 8be6a68 load on device 'cuda:0' took 2.23 ms  (cached)


In [14]:
exportSimulationSystem(exportPath, f'finalState', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**extraData, **{
    'kineticEnergy': kineticEnergy,
    'thermalEnergy': thermalEnergy,
    'totalEnergy': totalEnergy,
    'frame_num': i,
}))

In [15]:
ffmpeg_cmd = "ffmpeg -y -loglevel error -hide_banner -framerate 50 -f image2 -pattern_type glob -i 'frame_*.png' -c:v libx264 -pix_fmt yuv420p -b:v 10M output.mp4"
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4  -vf "fps=50,scale=540:-1:flags=lanczos,palettegen" palette.png'
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4 -i palette.png -filter_complex "fps=25,scale=540:-1:flags=lanczos[x];[x][1:v]paletteuse" out.gif'
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)

# now copy the output.mp4 and out.gif to the parent directory for easier access
shutil.copy(f'{imagePath}/output.mp4', f'{exportPath}/output.mp4')
shutil.copy(f'{imagePath}/out.gif', f'{exportPath}/out.gif');